# Failure Mode 7: Repeated Action Loop

> Before starting, read the [project README](../../README.md) for setup instructions and background on traces, scorers, and failure modes.

The agent gets stuck repeating actions without making progress toward completing the task. This covers three sub-cases: an **error retry loop** (the same failing tool call repeated), **cyclical alternation** (two or more tools looping without advancing), and a **semantic loop** (different tools chasing the same failed goal without changing strategy).

We evaluate this using two approaches:

| Scorer | Source | Needs expectations? | What it checks |
|---|---|---|---|
| Custom `@scorer` | MLflow native | No | Deterministic detection of retry loops and cyclical tool-call patterns |
| Custom `make_judge()` | MLflow native | No | LLM judge assessment of whether the agent is making progress or repeating a failed goal |

For a detailed explanation of this failure mode and the scorers, see [repeated_action_loop.md](repeated_action_loop.md).

### Prerequisites and setup

Start a local MLflow server before running this notebook:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

In [ ]:
import json
import sys
from pathlib import Path
from typing import Literal

sys.path.insert(0, str(Path("../..").resolve()))

import mlflow
from mlflow.entities import Feedback, SpanType, Trace
from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import scorer
from tools import TRAVEL_AGENT_TOOLS
from utils import print_eval_results

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("agentic-evaluation")
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name("agentic-evaluation")

# Clean up old traces for this failure mode
client = mlflow.MlflowClient()
old_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'repeated_action_loop'",
    return_type="list",
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old traces.")

### Scenario setup

This notebook uses two different user requests so the trace inputs match the travel scenarios exactly:

- **NYC -> Atlantis:** used for the error retry loop and semantic loop, where the destination is impossible
- **BOS -> SFO:** used for the cyclical alternation and normal progression traces, where flights exist but the agent still may or may not make progress

In [ ]:
RETRY_USER_MSG = [
    {
        "role": "user",
        "content": "Book me a flight from NYC to Atlantis on 2026-08-15.",
    }
]

CYCLE_USER_MSG = [
    {
        "role": "user",
        "content": "Book me a flight from Boston to San Francisco on 2026-08-15.",
    }
]

SEMANTIC_USER_MSG = [
    {
        "role": "user",
        "content": "Book me a flight from NYC to Atlantis on 2026-08-15.",
    }
]

PASS_USER_MSG = [
    {
        "role": "user",
        "content": "Book me a flight from Boston to San Francisco on 2026-08-15.",
    }
]

### Create traces

We create four synthetic traces simulating a travel agent handling two itineraries:

- **Error retry loop:** `search_flights` is called three times with identical arguments, failing the same way each time. Instead of retrying, the agent should tell the user no flights were found and ask whether they want to confirm or change the destination.
- **Cyclical alternation:** `search_flights` and `get_flight_details` alternate twice without progressing to a booking. Instead of repeating the same lookup cycle, the agent should either book the identified flight or ask the user whether they want a different option.
- **Semantic loop:** The agent alternates between `search_flights` and `search_alternative_routes` for the same failed goal, without changing strategy. Instead of pursuing the same impossible route, the agent should explain the failure and ask whether the user wants to change the destination, date, or routing constraints.
- **Normal progression (pass):** The agent searches, reviews details, and books the flight with no repetition.

In [ ]:
# --- Error retry loop (fail) ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def repeated_action_loop_retry(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "repeated_action_loop", "expected_result": "fail"}
    )

    for _ in range(3):
        with mlflow.start_span(name="search_flights", span_type=SpanType.TOOL) as span:
            span.set_inputs({
                "from_city": "NYC",
                "to_city": "Atlantis",
                "date": "2026-08-15",
            })
            span.set_outputs({"error": "no flights available"})

    return "I'm still checking flights to Atlantis."


# --- Cyclical alternation (fail) ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def repeated_action_loop_cycle(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "repeated_action_loop", "expected_result": "fail"}
    )

    for span_name, inputs, outputs in [
        (
            "search_flights",
            {"from_city": "BOS", "to_city": "SFO", "date": "2026-08-15"},
            [{"flight_id": "F100"}],
        ),
        (
            "get_flight_details",
            {"flight_id": "F100"},
            {"flight_id": "F100", "price": 450},
        ),
        (
            "search_flights",
            {"from_city": "BOS", "to_city": "SFO", "date": "2026-08-15"},
            [{"flight_id": "F100"}],
        ),
        (
            "get_flight_details",
            {"flight_id": "F100"},
            {"flight_id": "F100", "price": 450},
        ),
    ]:
        with mlflow.start_span(name=span_name, span_type=SpanType.TOOL) as span:
            span.set_inputs(inputs)
            span.set_outputs(outputs)

    return "I'm still comparing flight options."


# --- Semantic loop (fail) ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def repeated_action_loop_semantic(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "repeated_action_loop", "expected_result": "fail"}
    )

    for span_name, inputs, outputs in [
        (
            "search_flights",
            {"from_city": "NYC", "to_city": "Atlantis", "date": "2026-08-15"},
            {"error": "no flights available"},
        ),
        (
            "search_alternative_routes",
            {"from_city": "NYC", "to_city": "Atlantis", "date": "2026-08-15"},
            {"error": "no alternative routes found"},
        ),
        (
            "search_flights",
            {"from_city": "NYC", "to_city": "Atlantis", "date": "2026-08-15"},
            {"error": "no flights available"},
        ),
    ]:
        with mlflow.start_span(name=span_name, span_type=SpanType.TOOL) as span:
            span.set_inputs(inputs)
            span.set_outputs(outputs)

    return "I'm trying another way to find flights to Atlantis."


# --- Normal progression (pass) ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def repeated_action_loop_pass(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "repeated_action_loop", "expected_result": "pass"}
    )

    for span_name, inputs, outputs in [
        (
            "search_flights",
            {"from_city": "BOS", "to_city": "SFO", "date": "2026-08-15"},
            [{"flight_id": "F200"}],
        ),
        (
            "get_flight_details",
            {"flight_id": "F200"},
            {"flight_id": "F200", "price": 410},
        ),
        (
            "book_flight",
            {"flight_id": "F200"},
            {"booking_id": "BK-200", "status": "confirmed"},
        ),
    ]:
        with mlflow.start_span(name=span_name, span_type=SpanType.TOOL) as span:
            span.set_inputs(inputs)
            span.set_outputs(outputs)

    return "I found a flight, reviewed the details, and booked it successfully."


repeated_action_loop_retry(RETRY_USER_MSG)
repeated_action_loop_cycle(CYCLE_USER_MSG)
repeated_action_loop_semantic(SEMANTIC_USER_MSG)
repeated_action_loop_pass(PASS_USER_MSG)
mlflow.flush_trace_async_logging()
print("Created 4 traces (3 fail, 1 pass)")

### Load traces

We fetch the Repeated Action Loop traces we just created:
- **Error retry loop:** `search_flights` repeated three times with an identical failing result
- **Cyclical alternation:** `search_flights` → `get_flight_details` → `search_flights` → `get_flight_details`
- **Semantic loop:** `search_flights` → `search_alternative_routes` → `search_flights`, same goal, no progress
- **Normal progression:** `search_flights` → `get_flight_details` → `book_flight`

In [ ]:
repeated_action_loop_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'repeated_action_loop'",
    return_type="list",
)

print(f"Traces found: {len(repeated_action_loop_traces)}")
for t in repeated_action_loop_traces:
    tags = t.info.tags or {}
    print(
        f"  [{tags.get('expected_result', '?')}] Request: {str(t.info.request_preview)[:80]}"
    )
    print(f"    Response: {str(t.info.response_preview)[:80]}")
    print()

### Define scorers

- **Deterministic `@scorer`:** Inspects the tool spans in a trace directly. It first looks for three or more identical consecutive tool calls (same name, inputs, and outputs) to catch an error retry loop, then looks for back-to-back repeated sequences of matching tool signatures to catch cyclical alternation.
- **`make_judge()` semantic loop check:** An LLM judge reads the full trace and determines whether the agent is making progress toward the user's goal, even if it switches tools or approaches along the way.

In [ ]:
@scorer
def repeated_action_loop_deterministic(*, trace: Trace) -> Feedback:
    tool_spans = list(trace.search_spans(span_type=SpanType.TOOL))
    retry_streak = 1
    longest_retry_streak = 1

    def normalize(value: object) -> str:
        return json.dumps(value, sort_keys=True, default=str)

    for previous, current in zip(tool_spans, tool_spans[1:], strict=False):
        same_name = previous.name == current.name
        same_inputs = normalize(previous.inputs) == normalize(current.inputs)
        same_outputs = normalize(previous.outputs) == normalize(current.outputs)
        if same_name and same_inputs and same_outputs:
            retry_streak += 1
            longest_retry_streak = max(longest_retry_streak, retry_streak)
        else:
            retry_streak = 1

    if longest_retry_streak >= 3:
        return Feedback(
            value="no",
            rationale=f"Error retry loop detected: identical tool call repeated {longest_retry_streak} times.",
        )

    tool_signatures = [
        (span.name, normalize(span.inputs), normalize(span.outputs))
        for span in tool_spans
    ]
    for pattern_length in range(2, len(tool_signatures) // 2 + 1):
        for start in range(0, len(tool_signatures) - pattern_length * 2 + 1):
            pattern = tool_signatures[start : start + pattern_length]
            repeated = tool_signatures[
                start + pattern_length : start + pattern_length * 2
            ]
            if pattern == repeated:
                pattern_names = [name for name, _, _ in pattern]
                return Feedback(
                    value="no",
                    rationale=(
                        "Cyclical alternation detected: "
                        f"{pattern_names} repeated consecutively with matching inputs and outputs."
                    ),
                )

    return Feedback(
        value="yes",
        rationale="No retry loop or cyclical alternation detected in the tool trajectory.",
    )


semantic_loop_judge = make_judge(
    name="semantic_loop_check",
    instructions=(
        "Look at the sequence of tool calls in {{ trace }}. "
        "Determine if the agent is making progress toward the user's goal "
        "or repeating the same failed objective using different tools or approaches. "
        "If the agent tried the same goal multiple times without changing "
        "strategy or informing the user, return 'no'. Otherwise return 'yes'."
    ),
    model="openai:/gpt-4o",
    feedback_value_type=Literal["yes", "no"],
)

### Approach 1: Deterministic pattern detection (custom `@scorer`)

`repeated_action_loop_deterministic` reads the trace's tool spans directly and applies two checks: identical consecutive tool calls with the same inputs and outputs (error retry loop) and repeating sequences of matching tool signatures (cyclical alternation). No LLM or expectations needed -- fast and reproducible.

In [ ]:
with mlflow.start_run(run_name="repeated-action-loop-deterministic") as run:
    results_deterministic = mlflow.genai.evaluate(
        data=repeated_action_loop_traces,
        scorers=[repeated_action_loop_deterministic],
    )

print_eval_results(
    results_deterministic,
    "repeated_action_loop_deterministic",
    EXPERIMENT.experiment_id,
)

### Approach 2: Semantic loop detection (custom `make_judge()`)

`semantic_loop_check` reads the full trace and reasons about whether the agent is making progress toward the user's goal. This catches the semantic loop case -- different tools, same failed goal -- that a purely structural check might miss. Because it is an LLM-based judge, it requires an API key and its verdict can vary slightly across runs or judge models.

In [ ]:
with mlflow.start_run(run_name="repeated-action-loop-semantic-judge") as run:
    results_semantic = mlflow.genai.evaluate(
        data=repeated_action_loop_traces,
        scorers=[semantic_loop_judge],
    )

print_eval_results(results_semantic, "semantic_loop_check", EXPERIMENT.experiment_id)

### Interpreting the results

- Error retry loop -> both approaches say `no`
- Cyclical alternation -> both approaches say `no`
- Semantic loop -> the deterministic scorer says `yes` (it only catches exact repeated calls and repeated sequences with matching inputs and outputs), while `make_judge()` says `no` (it reasons about the goal, not just the tool-call structure)
- Normal progression -> both approaches say `yes`

**When to use which:**
- **Deterministic `@scorer` (Approach 1):** Best for CI/CD -- fast, reproducible, no LLM cost. Reliably catches error retry loops and cyclical alternation by inspecting exact tool-call structure directly. Misses semantic loops where the agent switches tools or keeps pursuing the same failed goal with different tool signatures.
- **`make_judge()` semantic check (Approach 2):** Best for catching subtler loops, including semantic ones, where the agent appears to be doing something different but is not actually making progress toward the goal. Requires an LLM API key and is non-deterministic, so results can vary slightly between runs or judge models.

For full details on this failure mode and both approaches, see [repeated_action_loop.md](repeated_action_loop.md).